# Alignment pipeline (debug, fixed)

Robust discovery of episodes_audio by scanning upward through parent directories. Prints cwd, chosen episodes dir and found .webm files.

In [1]:
!git clone https://github.com/Yi-Star32/Video_Analyzer.git

Cloning into 'Video_Analyzer'...
remote: Enumerating objects: 35, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 35 (delta 2), reused 29 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (35/35), 37.67 KiB | 4.71 MiB/s, done.
Resolving deltas: 100% (2/2), done.


In [1]:
%cd Video_Analyzer

/content/Video_Analyzer


In [3]:
!pip install -r requirements.txt

ERROR: Ignored the following versions that require a different python version: 0.1.0 Requires-Python >=3.13; 0.2.0 Requires-Python >=3.13; 0.2.1 Requires-Python >=3.13; 0.2.2 Requires-Python >=3.13
ERROR: Could not find a version that satisfies the requirement audioop-lts==0.2.2 (from versions: none)
ERROR: No matching distribution found for audioop-lts==0.2.2


In [4]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 91.0 MB/s eta 0:00:00


In [5]:
!pip install git+https://github.com/m-bain/whisperx.git

  Cloning https://github.com/m-bain/whisperx.git to /tmp/pip-req-build-4tp36lq1
  Running command git clone --filter=blob:none --quiet https://github.com/m-bain/whisperx.git /tmp/pip-req-build-4tp36lq1
  Resolved https://github.com/m-bain/whisperx.git to commit 3ccc17b8de34f305300f8a3fd3c9f76ba820c0d0
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.0 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of transformers to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of transformers to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with str

In [2]:
from audio_matcher import alignment, chunking, embedding, index, io, phonemes

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import sys
from pathlib import Path
import warnings
from tqdm import TqdmWarning

# 1. Zorg dat Python de 'audio_matcher' map kan vinden
# (Pas dit pad aan als je Video_Analyzer map ergens anders staat, bv. in je Drive)
project_path = "/content/Video_Analyzer"
if project_path not in sys.path:
    sys.path.append(project_path)

# 2. Onderdruk specifieke tqdm waarschuwingen
warnings.filterwarnings("ignore", category=TqdmWarning)

# 3. Importeer de audio_matcher onderdelen
from audio_matcher.embedding import AudioEmbeddingPipeline
from audio_matcher.phonemes import PhonemeAligner
from audio_matcher.alignment import build_phoneme_index_from_episodes, run_phoneme_pipeline
from audio_matcher.io import export_audio

# 4. Stel het absolute pad in naar je song (zorg dat je Drive gekoppeld is!)
SONG_PATH = Path("/content/drive/MyDrive/projecten/Video_Analyzer_data/output/separated/htdemucs/audio/vocals.wav")

# 5. Check direct of het bestand bestaat
if SONG_PATH.exists():
    print("🎉 Alles succesvol geïmporteerd en het audiobestand is gevonden!")
else:
    print("⚠️ Imports geslaagd, maar het bestand 'vocals.wav' werd niet gevonden op dit pad. Check of je Drive is gekoppeld.")

🎉 Alles succesvol geïmporteerd en het audiobestand is gevonden!


In [5]:
from pathlib import Path

# Pas dit aan naar de exacte plek van je projectmap
# Als hij in je Drive staat: Path("/content/drive/MyDrive/.../Video_Analyzer")
repo_root = Path("/content/drive/MyDrive/projecten/Video_Analyzer_data")

print('notebook cwd ingesteld op =', repo_root)
EPISODES_DIR = None

# Vanaf hier blijft je eigen code exact hetzelfde:
for p in [repo_root] + list(repo_root.parents):
    candidates = [
        p / 'data' / 'episodes_audio',
        p / 'audio_matcher' / 'data' / 'episodes_audio',
        p / 'episodes_audio',
    ]
    for c in candidates:
        if c.exists():
            EPISODES_DIR = c
            break
    if EPISODES_DIR is not None:
        break

if EPISODES_DIR is None:
    EPISODES_DIR = repo_root / 'data' / 'episodes_audio'

print('EPISODES_DIR chosen:', EPISODES_DIR)
if not EPISODES_DIR.exists():
    print('EPISODES_DIR does not exist:', EPISODES_DIR)
    files = []
else:
    files = sorted(EPISODES_DIR.rglob('audio.wav'))
    print(f'Found {len(files)} audio.wav files under {EPISODES_DIR}')
    for f in files:
        print('-', f)

notebook cwd ingesteld op = /content/drive/MyDrive/projecten/Video_Analyzer_data
EPISODES_DIR chosen: /content/drive/MyDrive/projecten/Video_Analyzer_data/data/episodes_audio
Found 30 audio.wav files under /content/drive/MyDrive/projecten/Video_Analyzer_data/data/episodes_audio
- /content/drive/MyDrive/projecten/Video_Analyzer_data/data/episodes_audio/1TlOcjJodHw/audio.wav
- /content/drive/MyDrive/projecten/Video_Analyzer_data/data/episodes_audio/326R_Lhua5w/audio.wav
- /content/drive/MyDrive/projecten/Video_Analyzer_data/data/episodes_audio/3l1lSNQxJA0/audio.wav
- /content/drive/MyDrive/projecten/Video_Analyzer_data/data/episodes_audio/6uNYmqvF24k/audio.wav
- /content/drive/MyDrive/projecten/Video_Analyzer_data/data/episodes_audio/9IymXcXl4fk/audio.wav
- /content/drive/MyDrive/projecten/Video_Analyzer_data/data/episodes_audio/9aLD8SGoPIc/audio.wav
- /content/drive/MyDrive/projecten/Video_Analyzer_data/data/episodes_audio/9wAKxOavTnw/audio.wav
- /content/drive/MyDrive/projecten/Video_A

In [ ]:
import gc
import torch
import numpy as np

# Automatisch GPU kiezen als die er is, anders veilig terugvallen op CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 WhisperX gaat draaien op: {device.upper()}")

pipeline = AudioEmbeddingPipeline()

# Hier vullen we de 'device' variabele in
aligner = PhonemeAligner(device=device, whisper_model='base')

# Je lijst met bestanden uit de vorige cell
files = files

# Start het bouwen van de index
pindex = build_phoneme_index_from_episodes(files, aligner, pipeline)

print("🎉 De phoneme index is succesvol opgebouwd!")

🚀 WhisperX gaat draaien op: CUDA


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

  [audio.wav] 9128 phonemes
2026-06-02 11:07:27 - whisperx.asr - INFO - Detected language: en (0.80) in first 30s of audio



Building phoneme index from episodes:  40%|████      | 12/30 [06:13<09:11, 30.65s/it]

  [audio.wav] 8334 phonemes
2026-06-02 11:07:57 - whisperx.asr - INFO - Detected language: en (0.82) in first 30s of audio



Building phoneme index from episodes:  43%|████▎     | 13/30 [06:42<08:31, 30.08s/it]

  [audio.wav] 7564 phonemes
2026-06-02 11:08:25 - whisperx.asr - INFO - Detected language: en (0.82) in first 30s of audio


In [ ]:
# final: run phoneme pipeline on chosen song and export
if not files:
    print('No reference files found, skipping pipeline')
else:
    final_audio = run_phoneme_pipeline(SONG_PATH, None, aligner, pipeline, pindex=pindex)
    print(f"output_ms: {len(final_audio)}")
    export_audio(final_audio, '/content/drive/MyDrive/projecten/Video_Analyzer_data/output/aligned_output_colab1.wav')
    print('Wrote aligned_output.wav')


NameError: name 'pindex' is not defined